In [1]:
import sys
sys.path.append("..")
from src.features import *
from src.data.make_dataset import *
%load_ext autoreload
%autoreload 2


c:\Users\ziacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(
c:\Users\ziacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


# Running the Zoning Nonconformity Analysis
Base Zoning

Issues: 
Watertown: Missing PSCD Zones




In [3]:
mmc_munis = ["Arlington", 
            "Braintree", "Brookline", "Cambridge",
              "Chelsea", 
             "Everett", "Lynn", "Malden", "Melrose",
             "Medford","Newton", "Revere", "Somerville", "Quincy", 
              "Watertown", "Winthrop"]

output_gdb = r"\\Data-Sync\Public\DataServices\Projects\Current_Projects\Housing\Zoning-to-Built-Form\Rightsizing Zoning\Rightsizing Zoning.gdb"

gdf_list = []

Base Parcel Scoring

In [4]:
from datetime import datetime
from src.data.weights import *
from src.features.zoning_nonconformity_scripts import *

for muni in mmc_munis:
    print(muni + ' processing starting at ' + str(datetime.now()))



    conf_output = run_zoning_nonconformity(muni, 'base')
    gdf_list.append(conf_output)
    outputlayername = muni + "_conformityscores_base"
    print(outputlayername)
    gdb_write(gdf = conf_output, gdb = output_gdb, layer_name = outputlayername)

all_munis = pd.concat(gdf_list)
gdb_write(gdf = all_munis, gdb = output_gdb, layer_name = "mmc_base")


Arlington processing starting at 2025-12-15 10:50:17.083252


URLError: <urlopen error [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond>

Overlay Scoring

In [5]:

from datetime import datetime
from src.data.weights import *
from src.features.zoning_nonconformity_scripts import *

gdf_list_overlay = []

for muni in mmc_munis:
    print(muni + ' processing starting at ' + str(datetime.now()))

    conf_output = run_zoning_nonconformity(muni, 'overlay')
    if len(conf_output) == 0:
        continue
    else:
        gdf_list_overlay.append(conf_output)
        outputlayername = muni + "_conformityscores_overlay"
        print(outputlayername)
        gdb_write(gdf = conf_output, gdb = output_gdb, layer_name = outputlayername)

all_munis_overlay = pd.concat(gdf_list_overlay)
gdb_write(gdf = all_munis_overlay, gdb = output_gdb, layer_name = "mmc_overlay")

Arlington processing starting at 2025-12-15 09:37:04.265774


URLError: <urlopen error [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond>

Base Test Results

In [ ]:
from datetime import datetime
from src.data.weights import *
from src.features.zoning_nonconformity_scripts import *

for muni in mmc_munis:
    print(muni + ' processing starting at ' + str(datetime.now()))



    test_results = run_zoning_nonconformity(muni, 'base', score = False)
    
    print(muni)
    print(test_results)

Base Zone Scoring

In [ ]:
from datetime import datetime
from src.data.weights import *
from src.features.zoning_nonconformity_scripts import *
gdf_list_diss = []

for muni in mmc_munis:
    print(muni + ' processing starting at ' + str(datetime.now()))

    conf_output = run_zoning_nonconformity(muni, 'base')
    
    conf_by_zone = conf_output[['ZO_CODE', 'CITY','size_sum', 'shape_sum', 'dense_sum',
                              'size_count', 'shape_count', 'dense_count',
                              'Total', 'Measures', 'geometry']].dissolve(by = ['ZO_CODE', 'CITY'],
                                       aggfunc = 'mean')
    gdf_list_diss.append(conf_by_zone)

all_munis_by_zone = pd.concat(gdf_list_diss)

gdb_write(gdf = all_munis_by_zone, gdb = output_gdb, layer_name = 'mmc_base_zone_scores')


Results of Analysis

In [29]:
from src.features.zoning_nonconformity_scripts import *
summary_list = []
excel_file = r"C:\Users\ziacovino\OneDrive - Metropolitan Area Planning Council\Metro Mayors Housing Task Force\Phase 2 Scope of Work\Rightsizing Zoning Project\Data\Analysis Outputs\summary_conformity10.17.xlsx"

for muni in mmc_munis:
    print(muni + ' processing starting at ' + str(datetime.now()))

    summary_output = analysis_outputs(muni, conf_summary= True)

    summary_list.append(summary_output)
    
   
    with pd.ExcelWriter(excel_file, mode = 'a') as writer:
        summary_output.to_excel(writer, sheet_name = muni)




full_summary_table = pd.concat(summary_list)

with pd.ExcelWriter(excel_file, mode = 'a') as writer:
         full_summary_table.to_excel(writer, sheet_name = "All MMC")



Arlington processing starting at 2025-10-17 14:57:50.636073
Braintree processing starting at 2025-10-17 14:57:52.203801
Cambridge processing starting at 2025-10-17 14:57:53.208474
Chelsea processing starting at 2025-10-17 14:57:55.853702
Everett processing starting at 2025-10-17 14:57:56.827699
Lynn processing starting at 2025-10-17 14:57:57.840007
Malden processing starting at 2025-10-17 14:57:59.213001
Melrose processing starting at 2025-10-17 14:58:02.330129
Medford processing starting at 2025-10-17 14:58:04.142135
Newton processing starting at 2025-10-17 14:58:05.252078
Revere processing starting at 2025-10-17 14:58:06.293073
Somerville processing starting at 2025-10-17 14:58:07.379074
Quincy processing starting at 2025-10-17 14:58:09.236584
Watertown processing starting at 2025-10-17 14:58:11.521212
Winthrop processing starting at 2025-10-17 14:58:14.561799


Summary Table Creation

In [20]:

all_munis_by_zone_table = pd.DataFrame(all_munis_by_zone.drop(columns = 'geometry'))
all_munis_by_zone_table['size_percent'] = all_munis_by_zone_table['size_sum']/all_munis_by_zone_table['size_count']
all_munis_by_zone_table['shape_percent'] = all_munis_by_zone_table['shape_sum']/all_munis_by_zone_table['shape_count']
all_munis_by_zone_table['dense_percent'] = all_munis_by_zone_table['dense_sum']/all_munis_by_zone_table['dense_count']

all_munis_by_zone_table.to_csv(r"C:\Users\ziacovino\OneDrive - Metropolitan Area Planning Council\Metro Mayors Housing Task Force\Phase 2 Scope of Work\Rightsizing Zoning Project\Data\Analysis Outputs.mmc_base_zone_scores.csv")

In [22]:
all_munis_table = pd.DataFrame(all_munis.drop(columns='geometry'))

all_munis_table.to_csv(r"C:\Users\ziacovino\OneDrive - Metropolitan Area Planning Council\Metro Mayors Housing Task Force\Phase 2 Scope of Work\Rightsizing Zoning Project\Data\Analysis Outputs.mmc_parcel_scores.csv")


Style Council

In [5]:
from datetime import datetime
from src.features.zoning_nonconformity_scripts import *
top_styles = []

for muni in mmc_munis:
    print(muni + ' processing starting at ' + str(datetime.now()))

    muni_styles = get_popular_building_styles(muni)
    muni_styles['Municipality'] = muni
    
    top_styles.append(muni_styles)

all_munis_styles = pd.concat(top_styles)

Arlington processing starting at 2025-11-04 09:53:53.612034
Braintree processing starting at 2025-11-04 09:54:03.009469
Brookline processing starting at 2025-11-04 09:54:07.947970
Cambridge processing starting at 2025-11-04 09:54:11.773742
Chelsea processing starting at 2025-11-04 09:54:16.744460
Everett processing starting at 2025-11-04 09:54:19.317200
Lynn processing starting at 2025-11-04 09:54:22.019532
Malden processing starting at 2025-11-04 09:54:26.635459
Melrose processing starting at 2025-11-04 09:54:29.757990
Medford processing starting at 2025-11-04 09:54:33.285241
Newton processing starting at 2025-11-04 09:54:38.883069


c:\Users\ziacovino\Desktop\rightsizing-zoning-analysis\notebooks\..\src\features\nested_functions.py:309: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  mapc_lpd = pd.read_csv(muni_lpd_path, dtype={


Revere processing starting at 2025-11-04 09:54:46.407805


c:\Users\ziacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
c:\Users\ziacovino\AppData\Local\ESRI\conda\envs\arcgispro-py3-clone\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: Geometry of polygon of fid 12717 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


Somerville processing starting at 2025-11-04 09:54:50.325992
Quincy processing starting at 2025-11-04 09:54:54.259232
Watertown processing starting at 2025-11-04 09:55:02.860369
Winthrop processing starting at 2025-11-04 09:55:06.513335


In [6]:
excel_file_style = r"C:\Users\ziacovino\OneDrive - Metropolitan Area Planning Council\Metro Mayors Housing Task Force\Phase 2 Scope of Work\Rightsizing Zoning Project\Data\Analysis Outputs\top_styles.xlsx"

all_munis_styles.to_excel(excel_file_style)